# Bacpipe Tutorial

This notebook walks through the main workflows of the `bacpipe` library — from inspecting configuration and loading audio, through generating embeddings with built-in and custom models, to running the full pipeline and benchmarking classifier performance.

---

**Contents**
1. [Setup & Configuration](#1.-Setup-&-Configuration)
2. [Loading Audio Files](#2.-Loading-Audio-Files)
3. [Ground Truth Labels](#3.-Ground-Truth-Labels)
4. [End-to-End: bacpipe.play](#4.-End-to-End:-bacpipe.play)
5. [Full Pipeline - Single Model](#5.-Full-Pipeline---Single-Model)
6. [Full Pipeline - Multiple Models](#6.-Full-Pipeline---Multiple-Models)
7. [Generating Embeddings - Full Directory (Save to Disk)](#7.-Generating-Embeddings---Full-Directory-(Save-to-Disk))
8. [Generating Embeddings - Single File (In-Memory)](#8.-Generating-Embeddings---Single-File-(In-Memory))
9. [High-Level Workflow - generate_embeddings](#9.-High-Level-Workflow---generate_embeddings)
10. [Benchmarking Classifier Performance](#10.-Benchmarking-Classifier-Performance)

---
## 0. Imports & Working Directory

Set the working directory to the repo root and import the display utility.

In [1]:
# to run successfully the packages for jupyter notebook need to be installed:
# uv pip install ipykernel, ipython

from IPython.display import display
import os
from pathlib import Path

# load the specific package
import bacpipe

/home/siriussound/Code/testing_repos/bacpipe/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/siriussound/Code/testing_repos/bacpipe/bacpipe/embedding_evaluation/visualization/dashboard.py:37: UserWarning: Using Panel interactively in VSCode notebooks requires the jupyter_bokeh package to be installed. You can install it with:

   pip install jupyter_bokeh

or:
    conda install jupyter_bokeh

and try again.
  pn.extension("plotly")


In [2]:
import importlib.resources as pkg_resources
os.chdir(pkg_resources.files("bacpipe"))
os.chdir('..')

# Change the value of the key main_results_dir in the namespace bacpipe.settings to change the directory 
# where the results of the tutorials are stored. By default, it is set to './bacpipe_results'.
bacpipe.settings.main_results_dir = str(Path(bacpipe.settings.main_results_dir) / 'simple_use_cases')

# !WARNING! the following code deletes the folder where the results of this tutorial is stored to be sure to start with a clean folder. 
# If you have important data in this folder, please comment it before running this code.
folder_path = bacpipe.settings.main_results_dir
if os.path.exists(folder_path):
    # Prompt the user
    user_input = input(f"Are you sure you want to delete '{folder_path}'? (y/n): ").lower().strip()

    if user_input == 'y':
        import shutil
        shutil.rmtree(folder_path) 
        print(f"Folder {folder_path} deleted.")
    else:
        print("Operation cancelled.")

else:
    print(f"Folder {folder_path} not found.")

Folder bacpipe_results/simple_use_cases deleted.


---
## 1. Setup & Configuration

Import `bacpipe` and inspect the current configuration and settings. You can also list all available API endpoints, supported models, and the embedding dimensions for each model.

In [3]:
import bacpipe

print('Config:')
display(bacpipe.config)

print('Settings:')
display(bacpipe.settings)

Config:


namespace(audio_dir='bacpipe/tests/test_data',
          overwrite=False,
          dashboard=True,
          models=['birdnet', 'birdnet_v3', 'perch_v2', 'perch_bird'],
          already_computed=False,
          dim_reduction_model='umap',
          evaluation_task=[])

Settings:


namespace(main_results_dir='bacpipe_results/simple_use_cases',
          embed_parent_dir='embeddings',
          dim_reduc_parent_dir='dim_reduced_embeddings',
          evaluations_dir='evaluations',
          model_base_path='bacpipe_model_checkpoints',
          device='cpu',
          global_batch_size=8,
          audio_suffixes=['.wav',
                          '.WAV',
                          '.aif',
                          '.mp3',
                          '.MP3',
                          '.flac',
                          '.ogg'],
          padding='wrap',
          avoid_pipelined_gpu_inference=False,
          nr_parallel_workers=False,
          rm_embedding_on_keyboard_interrupt=True,
          check_if_already_processed=True,
          check_if_already_dim_reduced=True,
          label_column='species',
          annotations_filename='annotations.csv',
          only_embed_annotations=False,
          min_annotation_length=0,
          default_label_keys=['time_of_d

In [4]:
# All available bacpipe API endpoints
bacpipe.__all__

['play',
 'run_pipeline_for_single_model',
 'run_pipeline_for_models',
 'generate_embeddings',
 'Loader',
 'Embedder',
 'get_audio_files',
 'DefaultLabels',
 'create_metadata_labels',
 'ground_truth_by_model',
 'get_metadata_labels',
 'get_dt_filename',
 'probing_pipeline',
 'run_probe_inference',
 'prepare_probe_inference',
 'clustering_pipeline',
 'run_clustering',
 'eval_clustering',
 'eval_with_silhouette',
 'benchmark',
 'model_specific_evaluation',
 'cross_model_evaluation',
 'confirm_model_name',
 'ensure_models_exist',
 'evaluation_with_settings_already_exists',
 'get_model_names',
 'make_set_paths_func',
 'visualize_using_dashboard',
 'supported_models',
 'models_needing_checkpoint',
 'TF_MODELS',
 'EMBEDDING_DIMENSIONS',
 'NEEDS_CHECKPOINT']

In [5]:
# All supported models
bacpipe.supported_models

['audiomae',
 'audioprotopnet',
 'avesecho_passt',
 'aves_especies',
 'bat',
 'batdetect2_clip_avg',
 'batdetect2_dets_avg',
 'beats',
 'birdaves_especies',
 'biolingual',
 'birdnet_v3',
 'birdnet',
 'birdmae',
 'convnext_birdset',
 'hbdet',
 'insect66',
 'insect459',
 'mix2',
 'naturebeats',
 'perch_bird',
 'perch_v2',
 'protoclr',
 'rcl_fs_bsed',
 'surfperch',
 'google_whale',
 'vggish']

In [6]:
# Embedding dimensions per model
bacpipe.EMBEDDING_DIMENSIONS

{'audiomae': 768,
 'audioprotopnet': 1024,
 'avesecho_passt': 768,
 'aves_especies': 768,
 'bat': 64,
 'batdetect2_clip_avg': 32,
 'batdetect2_dets_avg': 32,
 'beats': 768,
 'birdaves_especies': 1024,
 'biolingual': 512,
 'birdnet_v3': 1280,
 'birdnet': 1024,
 'birdmae': 1280,
 'convnext_birdset': 1024,
 'hbdet': 2048,
 'insect66': 1280,
 'insect459': 1280,
 'mix2': 960,
 'naturebeats': 768,
 'perch_bird': 1280,
 'perch_v2': 1536,
 'protoclr': 384,
 'rcl_fs_bsed': 2048,
 'surfperch': 1280,
 'google_whale': 1280,
 'vggish': 128}

---
## 2. Loading Audio Files

Retrieve all audio files from a directory as a list of strings.

`bacpipe.get_audio_files` recursively finds every audio file under the given
directory and returns the paths as a list. By default the paths are returned as
`pathlib.Path` objects; pass `return_type='str'` to get plain strings instead.
The supported file extensions are controlled by `bacpipe.settings.audio_suffixes`
(e.g. `.wav`, `.mp3`, `.flac`).


In [7]:
audio_files = bacpipe.get_audio_files(
    'bacpipe/tests/test_data', return_type='str'
)
audio_files

finding audio files: 11it [00:00, 25253.06it/s]
Found 7 number of audio files.


['bacpipe/tests/test_data/audio/FewShot/CHE_01_20190101_163410.wav',
 'bacpipe/tests/test_data/audio/FewShot/CHE_02_20190101_183410.wav',
 'bacpipe/tests/test_data/audio/FewShot/CHE_03_20190201_163410.wav',
 'bacpipe/tests/test_data/audio/FewShot/CHE_04_20190203_175410.wav',
 'bacpipe/tests/test_data/audio/UrbanSoundscape/242A2604603691DD_20250503_031300.WAV',
 'bacpipe/tests/test_data/audio/UrbanSoundscape/242A2604603691DD_20250503_031400.WAV',
 'bacpipe/tests/test_data/audio/UrbanSoundscape/242A2604603691DD_20250503_031500.WAV']

### 2.1 Extract datetime information from filenames

`bacpipe.get_dt_filename` extracts the timestamp encoded in a recording filename
and returns it as a `datetime.datetime` object. It relies on the common
bioacoustics naming convention *<prefix>_YYYYMMDD_HHMMSS.<ext>*, so a file like
`CHE_01_20190101_163410.wav` maps to `2019-01-01 16:34:10`. If no valid timestamp
can be parsed, the function logs a warning and falls back to the default datetime
`2000-01-01 00:00:00` rather than failing. This parsing is what powers the
metadata labels (`time_of_day`, `week_of_year`, ...) generated in the next section.


In [8]:
# Extract datetime information from audio filenames
dt_files = [bacpipe.get_dt_filename(file) for file in audio_files]
dt_files

[datetime.datetime(2019, 1, 1, 16, 34, 10),
 datetime.datetime(2019, 1, 1, 18, 34, 10),
 datetime.datetime(2019, 2, 1, 16, 34, 10),
 datetime.datetime(2019, 2, 3, 17, 54, 10),
 datetime.datetime(2025, 5, 3, 3, 13),
 datetime.datetime(2025, 5, 3, 3, 14),
 datetime.datetime(2025, 5, 3, 3, 15)]

---
## 3. Ground Truth Labels

Load multi-label ground truth annotations and align them to the model's timestamps.
Each row in the resulting array corresponds to the same time window as the model's
predictions, making it ready for evaluation.

Models such as BirdNET process audio in fixed-length context windows (e.g. 3
seconds). `bacpipe.ground_truth_by_model` reads `annotations.csv`, snaps the
annotations to that same time grid, and saves a `ground_truth.npy` file, so that
row *i* of the returned ground truth describes the exact same audio segment as
row *i* of the embeddings and predictions. Run it *after* the embeddings exist,
otherwise bacpipe cannot connect the embeddings to the labels.

`bacpipe.create_metadata_labels` generates a set of default labels for a
model/dataset combination, which is useful when no annotations are available.
The default labels are extracted from the filename if the format is
*<prefix>_YYYYMMDD_HHMMSS.<ext>* in order to get the *time_of_day*,
*day_of_year* and *continuous_timestamp*. The *parent directories* as well as the
*original files* are also part of the default labels. These labels are used by
the clustering and visualization steps.

By default, the results are saved in `bacpipe_results/test_data`. Two subfolders
are created: *embeddings* and *evaluations*. The *evaluations* folder is populated
with a subfolder per model (created by `ground_truth_by_model`) containing the
ground truth and the labels ready to be used by bacpipe.


In [9]:
# Load multi-label ground truth aligned to BirdNET's 3-second time bins
gt = bacpipe.ground_truth_by_model(
    model='birdnet',
    audio_dir='bacpipe/tests/test_data',
    annotations_filename='annotations.csv',
)
gt


No embeddings found for model birdnet in bacpipe_results/simple_use_cases/test_data/embeddings. Please check the directory path.
NoneType: None
No embeddings directory seems to exist. 
No embeddings found for model birdnet in bacpipe_results/simple_use_cases/test_data/embeddings. Please check the directory path.
2026-08-17 13:05:31.320373: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-17 13:05:31.356923: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
finding audio files: 11it [00:00, 29499.58it/s]
Found 7 number of audi

,start,end,audiofilename,simultaneous_labels,Eurasian Wren,Abbott's Babbler,Eurasian Blackbird,Common Cuckoo,Common Chaffinch,Eurasian Kestrel,Tree Pipit
0,0.0,3.0,bacpipe/tests/test_data/audio/FewShot/CHE_01_2...,1,0,0,0,0,0,0,1
1,3.0,6.0,bacpipe/tests/test_data/audio/FewShot/CHE_01_2...,1,0,0,0,0,0,0,1
2,6.0,9.0,bacpipe/tests/test_data/audio/FewShot/CHE_01_2...,1,0,0,0,0,0,0,1
3,9.0,12.0,bacpipe/tests/test_data/audio/FewShot/CHE_01_2...,2,0,0,0,0,0,1,1
4,12.0,15.0,bacpipe/tests/test_data/audio/FewShot/CHE_01_2...,1,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...
58,15.0,18.0,bacpipe/tests/test_data/audio/UrbanSoundscape/...,1,0,0,1,0,0,0,0
59,18.0,21.0,bacpipe/tests/test_data/audio/UrbanSoundscape/...,1,0,0,1,0,0,0,0
60,21.0,24.0,bacpipe/tests/test_data/audio/UrbanSoundscape/...,1,0,0,1,0,0,0,0
61,24.0,27.0,bacpipe/tests/test_data/audio/UrbanSoundscape/...,1,0,0,1,0,0,0,0


In [ ]:
# Generate metadata labels for a model/dataset combination
dl = bacpipe.metadata_labels(
    model='birdnet',
    audio_dir='bacpipe/tests/test_data'
)
dl


No embeddings found for model birdnet in bacpipe_results/simple_use_cases/test_data/embeddings. Please check the directory path.
NoneType: None
No embeddings found. Gathering files and nr of embeddings per file from audio files.
finding audio files: 11it [00:00, 37755.60it/s]
Found 7 number of audio files.
getting time of day: 100%|██████████| 7/7 [00:00<00:00, 90617.68it/s]
getting time per embeddings: 7it [00:00, 46455.90it/s]
getting day of year: 100%|██████████| 7/7 [00:00<00:00, 51239.32it/s]
getting continuous timestamps: 7it [00:00, 21812.87it/s]
getting parent directory: 7it [00:00, 24528.09it/s]
getting audio file names: 7it [00:00, 79782.96it/s]
Building metadata labels: 100%|██████████| 6/6 [00:00<00:00, 193.05it/s]


,time_of_day,week_of_year,day_of_year,continuous_timestamp,parent_directory,audio_file_name,start,end
0,16-34-10,2019--1,2019-01-01,2019-01-01_16:34:10,bacpipe/tests/test_data/audio/FewShot,bacpipe/tests/test_data/audio/FewShot/CHE_01_2...,0.0,3.0
1,16-34-13,2019--1,2019-01-01,2019-01-01_16:34:13,bacpipe/tests/test_data/audio/FewShot,bacpipe/tests/test_data/audio/FewShot/CHE_01_2...,3.0,6.0
2,16-34-16,2019--1,2019-01-01,2019-01-01_16:34:16,bacpipe/tests/test_data/audio/FewShot,bacpipe/tests/test_data/audio/FewShot/CHE_01_2...,6.0,9.0
3,16-34-19,2019--1,2019-01-01,2019-01-01_16:34:19,bacpipe/tests/test_data/audio/FewShot,bacpipe/tests/test_data/audio/FewShot/CHE_01_2...,9.0,12.0
4,16-34-22,2019--1,2019-01-01,2019-01-01_16:34:22,bacpipe/tests/test_data/audio/FewShot,bacpipe/tests/test_data/audio/FewShot/CHE_01_2...,12.0,15.0
...,...,...,...,...,...,...,...,...
58,03-15-15,2025--18,2025-05-03,2025-05-03_03:15:15,bacpipe/tests/test_data/audio/UrbanSoundscape,bacpipe/tests/test_data/audio/UrbanSoundscape/...,15.0,18.0
59,03-15-18,2025--18,2025-05-03,2025-05-03_03:15:18,bacpipe/tests/test_data/audio/UrbanSoundscape,bacpipe/tests/test_data/audio/UrbanSoundscape/...,18.0,21.0
60,03-15-21,2025--18,2025-05-03,2025-05-03_03:15:21,bacpipe/tests/test_data/audio/UrbanSoundscape,bacpipe/tests/test_data/audio/UrbanSoundscape/...,21.0,24.0
61,03-15-24,2025--18,2025-05-03,2025-05-03_03:15:24,bacpipe/tests/test_data/audio/UrbanSoundscape,bacpipe/tests/test_data/audio/UrbanSoundscape/...,24.0,27.0


---
## 4. End-to-End: `bacpipe.play`

`bacpipe.play` is the highest-level entry point. It runs the complete pipeline —
embeddings, classification, dimensionality reduction, evaluation, and an
interactive dashboard — in a single call. Ideal for a first exploration of a new
dataset across multiple models.

`play` does not return anything: all intermediate outputs are saved to disk and
the dashboard is launched automatically (unless disabled in the `config.yaml`
file). The models to run are passed with the `models` keyword (or taken from
`bacpipe.config.models`).

If a webpage with the dashboard is not automatically opened within your default
internet browser (e.g. chrome), copy then paste the URL displayed in the output
in a new page of your browser (e.g. http://localhost:5006).

The next cells select the device on which the models should run.
`bacpipe.settings.device` can be set to `'cpu'` (always works) or `'cuda'`
(only if a compatible NVIDIA GPU is available). If you do not have a GPU, leave
the default `'cpu'` to avoid CUDA errors.


In [11]:
bacpipe.settings.device='cuda'

In [12]:
# Get the cuDNN version
!conda list cudnn

zsh:1: command not found: conda


In [ ]:
bacpipe.play(
    models=['birdnet', 'perch_bird', 'naturebeats'],    # list of models to run. Supported models are in bacpipe.supported_models
    audio_dir='bacpipe/tests/test_data',                # path to directory containing audio files
    dim_reduction_model='umap',                          # dimensionality reduction model to use for visualization. Supported models are 'umap' and 'pca'. If 'None', no dimensionality reduction is applied.
)

Checking if the selected models require a checkpoint, and if so, if the checkpoint already exists.

birdnet checkpoint exists.

naturebeats checkpoint exists.

beats checkpoint exists.




###### Generating embeddings using BIRDNET ######

finding audio files: 11it [00:00, 28638.95it/s]
Found 7 number of audio files.
cuDNN version does not match the required 9.3 for tensorflow. Device is therefore set to cpu for the tensorflow models.
Using device='cuda'
2026-08-17 13:05:34.850414: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2026-08-17 13:05:34.850438: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:160] env: CUDA_VISIBLE_DEVICES="-1"
2026-08-17 13:05:34.850442: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:163] CUDA_VISIBLE_DEVICES is set to -1 - this hides all GPUs from CUDA
2026-08-17 13:05:34.850446

Launching server at http://localhost:5006


---
## 5. Full Pipeline - Single Model

`run_pipeline_for_single_model` runs the complete bacpipe pipeline for one model:
embedding generation, classifier inference, optional dimensionality reduction,
and visualisation. It returns a `Loader` object with all results accessible and
saves all intermediate outputs to disk.

Because the outputs are stored in bacpipe's predefined folder structure,
subsequent runs with the same model/dataset combination are much faster: bacpipe
detects the already-processed embeddings and skips recomputation automatically.

It does not compute any evaluations and does not launch the dashboard. It is
intended to be integrated into existing pipelines to return embeddings and
predictions for further processing.

- `model_name`: name of the model to run. Supported models are listed in
  `bacpipe.supported_models`.
- `audio_dir`: path to the directory containing the audio files.
- `dim_reduction_model`: dimensionality reduction model to use for visualization.
  Supported models are `'umap'` and `'pca'`; pass `'None'` (the default) to skip
  dimensionality reduction entirely.


In [14]:
loader_obj = bacpipe.run_pipeline_for_single_model(
    model_name='birdnet',                               # name of the model to run. Supported models are in bacpipe.supported_models
    audio_dir='bacpipe/tests/test_data',                # path to directory containing audio files  
    dim_reduction_model='umap'                          # dimensionality reduction model to use for visualization. Supported models are 'umap' and 'pca'. If 'None', no dimensionality reduction is applied. 
)
loader_obj




###### Generating embeddings using BIRDNET ######

INFO:bacpipe:


###### Generating embeddings using BIRDNET ######

finding audio files: 11it [00:00, 36044.80it/s]
Found 7 number of audio files.
INFO:bacpipe:Found 7 number of audio files.
cuDNN version does not match the required 9.3 for tensorflow. Device is therefore set to cpu for the tensorflow models.
INFO:bacpipe:cuDNN version does not match the required 9.3 for tensorflow. Device is therefore set to cpu for the tensorflow models.
Using device='cuda'
INFO:bacpipe:Using device='cuda'
Skipping model.eval() because model is from tensorflow.
ERROR:bacpipe:Skipping model.eval() because model is from tensorflow.
 processing batches: 100%|██████████| 1/1 [00:00<00:00,  4.79it/s]2026-08-17 13:06:16.605699: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
                                                                  


###### Generating embeddings using U

---
## 6. Full Pipeline - Multiple Models

`run_pipeline_for_models` does the same as `run_pipeline_for_single_model`, but
runs the full pipeline across a list of models in one call. It returns a
dictionary of `Loader` objects keyed by model name. This makes it straightforward
to compare embeddings and predictions across models on the same dataset.

Each `Loader` exposes the same methods as above — e.g. `.metadata_dict`,
`.embeddings()` and `.predictions()`.


In [15]:
loader_dictionary = bacpipe.run_pipeline_for_models(
    models=['birdnet', 'naturebeats'],                  # list of models to run. Supported models are in bacpipe.supported_models
    audio_dir='bacpipe/tests/test_data',                # path to directory containing audio files  
    dim_reduction_model='umap'                          # dimensionality reduction model to use for visualization. Supported models are 'umap' and 'pca'. If 'None', no dimensionality reduction is applied. 
)

display(loader_dictionary['birdnet'].metadata_dict)
display(loader_dictionary['naturebeats'].embeddings())




###### Generating embeddings using BIRDNET ######

INFO:bacpipe:


###### Generating embeddings using BIRDNET ######

finding audio files: 11it [00:00, 26622.82it/s]
Found 7 number of audio files.
INFO:bacpipe:Found 7 number of audio files.
cuDNN version does not match the required 9.3 for tensorflow. Device is therefore set to cpu for the tensorflow models.
INFO:bacpipe:cuDNN version does not match the required 9.3 for tensorflow. Device is therefore set to cpu for the tensorflow models.
Using device='cuda'
INFO:bacpipe:Using device='cuda'
Skipping model.eval() because model is from tensorflow.
ERROR:bacpipe:Skipping model.eval() because model is from tensorflow.
                                                                  


###### Generating embeddings using UMAP ######

INFO:bacpipe:


###### Generating embeddings using UMAP ######


### Embeddings already exist. Using embeddings in bacpipe_results/simple_use_cases/test_data/dim_reduced_embeddings/2026-08-17_13-05___umap-te

{'model_name': 'birdnet',
 'audio_dir': 'bacpipe/tests/test_data',
 'embed_dir': 'bacpipe_results/simple_use_cases/test_data/embeddings/2026-08-17_13-06___birdnet-test_data',
 'files': {'audio_files': ['audio/FewShot/CHE_01_20190101_163410.wav',
   'audio/FewShot/CHE_02_20190101_183410.wav',
   'audio/FewShot/CHE_03_20190201_163410.wav',
   'audio/FewShot/CHE_04_20190203_175410.wav',
   'audio/UrbanSoundscape/242A2604603691DD_20250503_031300.WAV',
   'audio/UrbanSoundscape/242A2604603691DD_20250503_031400.WAV',
   'audio/UrbanSoundscape/242A2604603691DD_20250503_031500.WAV'],
  'file_lengths (s)': [63.98977083333333,
   9.890729166666667,
   8.202479166666667,
   9.351708333333333,
   30.0,
   30.0,
   30.0],
  'nr_embeds_per_file': [22, 4, 3, 4, 10, 10, 10]},
 'segment_length (samples)': 144000,
 'sample_rate (Hz)': 48000,
 'embedding_size': 1024,
 'nr_embeds_total': 63,
 'total_dataset_length (s)': 181.4346875}

{'audio/FewShot/CHE_02_20190101_183410_naturebeats.npy': array([[-0.04508385, -0.27886578,  0.2902754 , ..., -0.05627377,
         -0.04705948, -0.22595635],
        [ 0.21691658, -0.01656353, -0.24687065, ..., -0.01271354,
         -0.15138532,  0.16802378]], dtype=float32),
 'audio/FewShot/CHE_04_20190203_175410_naturebeats.npy': array([[-0.07883446, -0.21604128,  0.0372471 , ..., -0.33621287,
         -0.4699472 ,  0.12062185],
        [ 0.09182488, -0.23457749, -0.15228029, ..., -0.5090017 ,
         -0.36838034, -0.01293317]], dtype=float32),
 'audio/FewShot/CHE_01_20190101_163410_naturebeats.npy': array([[ 0.02325631, -0.15567577,  0.22073437, ..., -0.22671428,
         -0.30680564,  0.50354177],
        [ 0.11346193, -0.18371455,  0.00739794, ..., -0.01205266,
         -0.30532116,  0.57489514],
        [-0.12632816, -0.16671248,  0.28616783, ...,  0.24787539,
         -0.5470489 ,  0.47346014],
        ...,
        [ 0.1841007 , -0.3042624 ,  0.05351426, ..., -0.05147238,
     

---
## 7. Generating Embeddings - Full Directory (Save to Disk)

Instead of using the all-in-one pipeline functions, you can also call the `Loader`
and `Embedder` classes directly for more control. They are the core building
blocks of bacpipe, used internally by `run_pipeline_for_single_model` and
`run_pipeline_for_models`.

To process all audio files in a directory and persist the results, pass
`use_folder_structure=True` to the `Loader`. Bacpipe will create a timestamped
output directory, run inference using multithreading, and save embeddings,
metadata, and classifier predictions. Subsequent runs detect the saved files and
skip recomputation automatically.

The `loader_obj` returned by the `Loader` is the same object type returned by
`run_pipeline_for_single_model` and `run_pipeline_for_models`, so you can use it
in the same way to access embeddings, metadata, and predictions:

- `loader_obj.metadata_dict`: an overview of the audio data, where the embeddings
  were saved, per-file embedding counts, the sample rate, the segment length, and
  the total processed duration.
- `loader_obj.embeddings(return_type='array')`: all embeddings concatenated into a
  2D array where axis 0 is the time/segment axis and axis 1 the embedding
  dimension.
- `loader_obj.predictions(...)`: the pretrained classifier outputs, either as a
  per-file dictionary, a concatenated array, or a dataframe.


In [16]:
MODEL_NAME = 'birdnet'

# Create a loader object that will handle all the audio file, path and parameters needed to compute the embeddings for instance
loader_obj = bacpipe.Loader(
    audio_dir='bacpipe/tests/test_data',
    model_name=MODEL_NAME,
    use_folder_structure=True
)

# Create an embededding object with the selected model (MODEL_NAME) passing the loader object in order to have the audio directory mapping
embed_obj = bacpipe.Embedder(
    model_name=MODEL_NAME, 
    loader=loader_obj)

# Process all files using multithreading
embed_obj.run_inference_pipeline_using_multithreading()

print('Metadata dict:')
display(loader_obj.metadata_dict)

print('Embeddings (array):')
display(loader_obj.embeddings(return_type='array'))

print('Predictions (array):')
display(loader_obj.predictions(return_type='array'))

print('Predictions (dataframe):')
display(loader_obj.predictions(return_type='dataframe'))

Finding all generated embeddings: 7it [00:00, 27235.74it/s]
Found 7 embedding files.
INFO:bacpipe:Found 7 embedding files.
finding audio files: 11it [00:00, 14100.66it/s]
Found 7 number of audio files.
INFO:bacpipe:Found 7 number of audio files.

### Embeddings already exist. Using embeddings in bacpipe_results/simple_use_cases/test_data/embeddings/2026-08-17_13-06___birdnet-test_data ###
INFO:bacpipe:
### Embeddings already exist. Using embeddings in bacpipe_results/simple_use_cases/test_data/embeddings/2026-08-17_13-06___birdnet-test_data ###
cuDNN version does not match the required 9.3 for tensorflow. Device is therefore set to cpu for the tensorflow models.
INFO:bacpipe:cuDNN version does not match the required 9.3 for tensorflow. Device is therefore set to cpu for the tensorflow models.
Using device='cuda'
INFO:bacpipe:Using device='cuda'
Skipping model.eval() because model is from tensorflow.
ERROR:bacpipe:Skipping model.eval() because model is from tensorflow.


Metadata dict:


{'audio_dir': 'bacpipe/tests/test_data',
 'embed_dir': 'bacpipe_results/simple_use_cases/test_data/embeddings/2026-08-17_13-06___birdnet-test_data',
 'embedding_size': 1024,
 'files': {'audio_files': ['audio/FewShot/CHE_01_20190101_163410.wav',
   'audio/FewShot/CHE_02_20190101_183410.wav',
   'audio/FewShot/CHE_03_20190201_163410.wav',
   'audio/FewShot/CHE_04_20190203_175410.wav',
   'audio/UrbanSoundscape/242A2604603691DD_20250503_031300.WAV',
   'audio/UrbanSoundscape/242A2604603691DD_20250503_031400.WAV',
   'audio/UrbanSoundscape/242A2604603691DD_20250503_031500.WAV'],
  'file_lengths (s)': [63.98977083333333,
   9.890729166666667,
   8.202479166666667,
   9.351708333333333,
   30.0,
   30.0,
   30.0],
  'nr_embeds_per_file': [22, 4, 3, 4, 10, 10, 10]},
 'model_name': 'birdnet',
 'nr_embeds_total': 63,
 'sample_rate (Hz)': 48000,
 'segment_length (samples)': 144000,
 'total_dataset_length (s)': 181.4346875}

Embeddings (array):


array([[0.        , 0.5192538 , 0.11016142, ..., 0.02030575, 0.        ,
        0.        ],
       [0.        , 0.04109726, 0.        , ..., 0.44500944, 0.30178508,
        1.4982907 ],
       [0.        , 0.04689805, 0.        , ..., 0.        , 0.        ,
        0.35709748],
       ...,
       [0.        , 0.36581063, 0.22898006, ..., 1.8080922 , 0.31529728,
        1.3595362 ],
       [0.        , 0.02800187, 0.02318055, ..., 0.48662075, 0.        ,
        0.610488  ],
       [0.        , 0.13184942, 0.3214704 , ..., 1.2065998 , 0.60497993,
        0.73578596]], dtype=float32)

Predictions (array):


(array([[0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.62529457],
        [0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.55551183],
        [0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.7379832 , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.8801123 ],
        [0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        ],
        [0.   

Predictions (dataframe):


,Unnamed: 0,audiofilename,start,end,simultaneous_labels,Short-toed Treecreeper,Eurasian Blackbird,Dunnock,Common Cuckoo
0,0,audio/FewShot/CHE_01_20190101_163410.wav,3.0,6.0,1,0.625295,0.000000,0.000000,0.000000
1,1,audio/FewShot/CHE_01_20190101_163410.wav,9.0,12.0,1,0.555512,0.000000,0.000000,0.000000
2,2,audio/FewShot/CHE_01_20190101_163410.wav,33.0,36.0,1,0.000000,0.000000,0.737983,0.000000
3,3,audio/FewShot/CHE_01_20190101_163410.wav,36.0,39.0,1,0.880112,0.000000,0.000000,0.000000
4,4,audio/FewShot/CHE_02_20190101_183410.wav,0.0,3.0,1,0.000000,0.000000,0.000000,0.604620
5,5,audio/FewShot/CHE_02_20190101_183410.wav,3.0,6.0,1,0.000000,0.000000,0.000000,0.817276
6,6,audio/FewShot/CHE_02_20190101_183410.wav,6.0,9.0,1,0.000000,0.000000,0.000000,0.606057
7,7,audio/FewShot/CHE_02_20190101_183410.wav,9.0,12.0,1,0.000000,0.000000,0.000000,0.703757
8,8,audio/FewShot/CHE_03_20190201_163410.wav,0.0,3.0,1,0.000000,0.000000,0.000000,0.510690
9,9,audio/FewShot/CHE_03_20190201_163410.wav,3.0,6.0,1,0.000000,0.000000,0.000000,0.584850


---
## 8. Generating Embeddings - Single File (In-Memory)

If you just want to generate embeddings for a single file without saving anything
to disk, instantiate the `Loader` and `Embedder` classes directly *without*
`use_folder_structure`. The embedding generation then runs in-memory and returns
Python objects without writing any files. This is useful for quick experimentation
or when you want to work with embeddings directly in memory.

Note that because nothing is saved to disk, `loader_obj.embeddings()` returns an
empty dict unless you keep a reference to the embedding arrays yourself (or save
them manually).


In [17]:
# Create a Loader without folder structure — nothing is written to disk
loader_obj = bacpipe.Loader('bacpipe/tests/test_data')
embed_obj = bacpipe.Embedder('birdnet', loader_obj)

# Generate embeddings for the first audio file
embeds = embed_obj.get_embeddings_from_model(loader_obj.files[0])

print('Since embeddings were not saved, .embeddings() returns empty:')
display(loader_obj.embeddings())

print('The embeddings are still accessible via the declared variable:')
display(embeds)

print('Classifier predictions are accessible through the embedder object:')
display(embed_obj.classifier.predictions)


finding audio files: 11it [00:00, 32908.23it/s]
Found 7 number of audio files.
INFO:bacpipe:Found 7 number of audio files.
No model_name is passed, therefore no directory structure will be created.
INFO:bacpipe:No model_name is passed, therefore no directory structure will be created.
cuDNN version does not match the required 9.3 for tensorflow. Device is therefore set to cpu for the tensorflow models.
INFO:bacpipe:cuDNN version does not match the required 9.3 for tensorflow. Device is therefore set to cpu for the tensorflow models.
Using device='cuda'
INFO:bacpipe:Using device='cuda'
Skipping model.eval() because model is from tensorflow.
ERROR:bacpipe:Skipping model.eval() because model is from tensorflow.
birdnet inference took 1.20s.                                     
INFO:bacpipe:birdnet inference took 1.20s.
No embedding files were found. Check that the path is right or if you actually have processed embeddings with None for file in bacpipe/tests/test_data.


Since embeddings were not saved, .embeddings() returns empty:


None

The embeddings are still accessible via the declared variable:


array([[0.        , 0.5192538 , 0.11016142, ..., 0.02030575, 0.        ,
        0.        ],
       [0.        , 0.04109726, 0.        , ..., 0.44500944, 0.30178508,
        1.4982907 ],
       [0.        , 0.04689805, 0.        , ..., 0.        , 0.        ,
        0.35709748],
       ...,
       [0.        , 0.62402207, 0.1901902 , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.28852096, 0.27240792, ..., 0.        , 0.        ,
        0.5682918 ],
       [0.        , 0.31145108, 0.18153723, ..., 0.        , 0.        ,
        0.45023617]], dtype=float32)

Classifier predictions are accessible through the embedder object:


tensor([[7.2337e-06, 1.2321e-06, 1.8781e-05,  ..., 9.8536e-06, 4.5944e-06,
         8.6071e-06],
        [2.6510e-06, 2.3206e-07, 8.3774e-06,  ..., 1.9020e-06, 1.8968e-06,
         4.7985e-07],
        [7.6319e-06, 1.2511e-05, 3.5127e-05,  ..., 5.9355e-05, 2.0990e-04,
         5.9185e-06],
        ...,
        [1.2915e-06, 1.7141e-07, 2.1810e-06,  ..., 1.5112e-04, 6.5344e-05,
         1.2531e-05],
        [1.5508e-05, 9.8838e-07, 7.9295e-07,  ..., 3.2521e-05, 2.3045e-05,
         1.1301e-05],
        [2.5794e-06, 3.8612e-06, 4.1147e-06,  ..., 4.6988e-05, 2.5489e-05,
         4.5167e-05]])

In [18]:
import numpy as np
print('Get the class label of the most highest values prediction:')
label_array = np.array(embed_obj.model.classes)
class_labels = label_array[np.argmax(embed_obj.classifier.predictions, axis=1)]
display(class_labels)

Get the class label of the most highest values prediction:


array(['Eurasian Penduline Tit', 'Short-toed Treecreeper', 'Dunnock',
       'Short-toed Treecreeper', 'Short-toed Treecreeper',
       'Common Chaffinch', 'Dunnock', 'Yellow-tufted Pipit',
       'Common Chaffinch', 'Flammulated Owl', 'Common Chaffinch',
       'Dunnock', 'Short-toed Treecreeper', 'Short-toed Treecreeper',
       'Verdin', 'Common Chaffinch', 'Dunnock',
       'Eurasian Three-toed Woodpecker', 'Common Chaffinch',
       'Lesser Nighthawk', 'Common Chaffinch', 'Rock Wren'], dtype='<U34')

---
## 9. High-Level Workflow - `generate_embeddings`

`bacpipe.generate_embeddings` wraps the `Loader`/`Embedder` pattern into a single
convenient call. It runs the embedding generation pipeline for one model —
including classification using the pretrained classifier when the model ships one —
and returns a `Loader` object that exposes the metadata and the computed
embeddings. By default the results are saved in the standard folder structure;
pass `use_folder_structure=False` to only save embeddings. 
Saving cannot be completely avoided with this pipelines, as keeping
everything in memory is not always feasible. Use the 
previously explained functions if you want to avoid saving
embeddings. The embeddings can then be retrieved with 
`loader_obj.embeddings(return_type='array')`.


In [ ]:
loader_obj = bacpipe.generate_embeddings(
    model_name='birdnet',
    audio_dir='bacpipe/tests/test_data'
)

display(loader_obj.metadata_dict)
display(loader_obj.embeddings(return_type='array'))




###### Generating embeddings using BIRDNET ######

INFO:bacpipe:


###### Generating embeddings using BIRDNET ######

Finding all generated embeddings: 7it [00:00, 14621.58it/s]
Found 7 embedding files.
INFO:bacpipe:Found 7 embedding files.
finding audio files: 11it [00:00, 33336.23it/s]
Found 7 number of audio files.
INFO:bacpipe:Found 7 number of audio files.

### Embeddings already exist. Using embeddings in bacpipe_results/simple_use_cases/test_data/embeddings/2026-08-17_13-06___birdnet-test_data ###
INFO:bacpipe:
### Embeddings already exist. Using embeddings in bacpipe_results/simple_use_cases/test_data/embeddings/2026-08-17_13-06___birdnet-test_data ###


{'audio_dir': 'bacpipe/tests/test_data',
 'embed_dir': 'bacpipe_results/simple_use_cases/test_data/embeddings/2026-08-17_13-06___birdnet-test_data',
 'embedding_size': 1024,
 'files': {'audio_files': ['audio/FewShot/CHE_01_20190101_163410.wav',
   'audio/FewShot/CHE_02_20190101_183410.wav',
   'audio/FewShot/CHE_03_20190201_163410.wav',
   'audio/FewShot/CHE_04_20190203_175410.wav',
   'audio/UrbanSoundscape/242A2604603691DD_20250503_031300.WAV',
   'audio/UrbanSoundscape/242A2604603691DD_20250503_031400.WAV',
   'audio/UrbanSoundscape/242A2604603691DD_20250503_031500.WAV'],
  'file_lengths (s)': [63.98977083333333,
   9.890729166666667,
   8.202479166666667,
   9.351708333333333,
   30.0,
   30.0,
   30.0],
  'nr_embeds_per_file': [22, 4, 3, 4, 10, 10, 10]},
 'model_name': 'birdnet',
 'nr_embeds_total': 63,
 'sample_rate (Hz)': 48000,
 'segment_length (samples)': 144000,
 'total_dataset_length (s)': 181.4346875}

array([[0.        , 0.5192538 , 0.11016142, ..., 0.02030575, 0.        ,
        0.        ],
       [0.        , 0.04109726, 0.        , ..., 0.44500944, 0.30178508,
        1.4982907 ],
       [0.        , 0.04689805, 0.        , ..., 0.        , 0.        ,
        0.35709748],
       ...,
       [0.        , 0.36581063, 0.22898006, ..., 1.8080922 , 0.31529728,
        1.3595362 ],
       [0.        , 0.02800187, 0.02318055, ..., 0.48662075, 0.        ,
        0.610488  ],
       [0.        , 0.13184942, 0.3214704 , ..., 1.2065998 , 0.60497993,
        0.73578596]], dtype=float32)

### 9.1 Load and concatenate audio files yourself, then pass the result to bacpipe

You can also load and preprocess audio yourself and pass it directly to an
`Embedder` — useful when you need custom loading logic or want to process audio
that isn't stored on disk.

Here we load all test audio files with `librosa`, concatenate the samples into a
single long 1D numpy array, and pass it to `Embedder.embeddings_using_multithreading`.
The method windows the audio into segments of the model's input length and returns
the embedding of each window. `bacpipe.ensure_models_exist` guarantees that the
model checkpoint is present locally, downloading it from the Hugging Face Hub on
first use.


In [20]:
# Alternatively: load and concatenate audio yourself, then pass it directly to an Embedder
import librosa as lb
import numpy as np

audio_files = bacpipe.get_audio_files(
    'bacpipe/tests/test_data', return_type='str'
)

audio = []
for file in audio_files:
    aud, sr = lb.load(file)
    audio.extend(aud)
audio = np.array(audio)

# check if the model exists, if not, download it
bacpipe.ensure_models_exist(bacpipe.settings.model_base_path, ['naturebeats'])
# embed
embed_obj = bacpipe.Embedder('naturebeats')
embeds = embed_obj.embeddings_using_multithreading(audio)
embeds

finding audio files: 11it [00:00, 24193.68it/s]
Found 7 number of audio files.
INFO:bacpipe:Found 7 number of audio files.
Checking if the selected models require a checkpoint, and if so, if the checkpoint already exists.

INFO:bacpipe:Checking if the selected models require a checkpoint, and if so, if the checkpoint already exists.

naturebeats checkpoint exists.

INFO:bacpipe:naturebeats checkpoint exists.

beats checkpoint exists.

INFO:bacpipe:beats checkpoint exists.

Using device='cuda'
INFO:bacpipe:Using device='cuda'
BEATs Config: {'input_patch_size': 16, 'embed_dim': 512, 'conv_bias': False, 'encoder_layers': 12, 'encoder_embed_dim': 768, 'encoder_ffn_embed_dim': 3072, 'encoder_attention_heads': 12, 'activation_fn': 'gelu', 'layer_wise_gradient_decay_ratio': 0.6, 'layer_norm_first': False, 'deep_norm': True, 'dropout': 0.0, 'attention_dropout': 0.0, 'activation_dropout': 0.0, 'encoder_layerdrop': 0.05, 'dropout_input': 0.0, 'conv_pos': 128, 'conv_pos_groups': 16, 'relative_pos

[array([-4.83361222e-02, -2.11153120e-01, -6.51885271e-02,  1.54393062e-01,
        -1.93824977e-01,  1.20965265e-01, -5.23431599e-01,  6.69790655e-02,
        -1.82799697e-01,  5.84068894e-01,  1.99384652e-02,  1.05700478e-01,
         2.06541449e-01,  7.54853249e-01, -5.18234491e-01,  2.84686178e-01,
        -3.19748253e-01, -1.65323228e-01,  5.33016443e-01, -5.15899003e-01,
        -2.02803358e-01, -3.62525314e-01, -1.18121393e-01,  1.44646943e-01,
         2.30197370e-01,  9.11206231e-02,  2.44908974e-01, -1.91879824e-01,
         7.18704909e-02,  8.88960715e-03,  2.03621313e-02, -5.12049913e-01,
        -1.74781367e-01,  8.41649920e-02, -1.79470912e-01,  1.34757012e-01,
        -5.40557951e-02, -1.13047875e-01,  1.38471007e-01,  1.67249724e-01,
         1.49837032e-01,  4.62806702e-01, -3.70645285e-01, -1.43066645e-01,
        -2.13485822e-01,  1.24435266e-02,  1.69104263e-02, -1.74495697e-01,
        -1.19829103e-01, -3.35719548e-02,  8.73068050e-02, -2.91838914e-01,
        -1.4

---
## 10. Benchmarking Classifier Performance

`bacpipe.benchmark` evaluates a model's pretrained classifier against your ground
truth annotations. It aligns predictions and ground truth to the same timestamps,
resolves label mismatches (for example hyphen or spacing differences such as
*Red-Shouldered Hawk* vs *Red Shouldered Hawk*) with a fuzzy-matching fallback,
and returns a per-species `sklearn` classification report with precision, recall,
and F1.

If predictions have already been generated for this dataset, the function loads
them from disk and runs very quickly.


In [21]:
results = bacpipe.benchmark(
    'birdnet',
    'bacpipe/tests/test_data',
    annotations_file='annotations.csv'
)
display(results)

Fetching ground truth and mapping it to model timestamps.

INFO:bacpipe:Fetching ground truth and mapping it to model timestamps.

Multiple embeddings found for model birdnet in bacpipe_results/simple_use_cases/test_data/embeddings. Using the most recent path.
INFO:bacpipe:Multiple embeddings found for model birdnet in bacpipe_results/simple_use_cases/test_data/embeddings. Using the most recent path.
The simultaneous labels column of the ground truth has values exceeding 1. This means you have multi-label ground truth annotations. If this should not be happening ensure the ground truth is created correcly.
The simultaneous labels column of the ground truth has values exceeding 1. This means you have multi-label ground truth annotations. If this should not be happening ensure the ground truth is created correcly.



###### Generating embeddings using BIRDNET ######

INFO:bacpipe:


###### Generating embeddings using BIRDNET ######

Finding all generated embeddings: 7it [00:00, 35246.25i

{'report': {'Common Cuckoo': {'precision': 1.0,
   'recall': 0.8181818181818182,
   'f1-score': 0.9,
   'support': 11.0},
  'Eurasian Blackbird': {'precision': 1.0,
   'recall': 0.6428571428571429,
   'f1-score': 0.782608695652174,
   'support': 28.0},
  'micro avg': {'precision': 1.0,
   'recall': 0.6923076923076923,
   'f1-score': 0.8181818181818182,
   'support': 39.0},
  'macro avg': {'precision': 1.0,
   'recall': 0.7305194805194806,
   'f1-score': 0.841304347826087,
   'support': 39.0},
  'weighted avg': {'precision': 1.0,
   'recall': 0.6923076923076923,
   'f1-score': 0.8157190635451506,
   'support': 39.0},
  'samples avg': {'precision': 0.6923076923076923,
   'recall': 0.6923076923076923,
   'f1-score': 0.6923076923076923,
   'support': 39.0}},
 'gt_binary': array([[1, 0],
        [1, 0],
        [1, 0],
        [1, 0],
        [1, 0],
        [1, 0],
        [1, 0],
        [1, 0],
        [1, 0],
        [1, 0],
        [1, 0],
        [0, 1],
        [0, 1],
        [0, 1]